# Generating the Analysis-Ready NHANES Dataset

This notebook is used to create the analysis-ready dataset for our research project. It loads the selected NHANES `.XPT` files from multiple survey cycles, merges them by participant identifier, harmonizes variables across cycles, constructs the diabetes outcome variable, applies the population criteria, and selects the final predictor variables.

The final output of this notebook is a cleaned and structured dataset that can be used in the next steps of the project, including preprocessing, train-test splitting, class imbalance handling, model training, evaluation, and feature-importance analysis.

**Note:** If the `DATASETS` folder already contains the generated data files, it is not necessary to run this notebook again. However, it is still useful to read through the notebook to understand how the dataset was created, which variables were selected, how the outcome variable was constructed, and how the final analysis-ready dataset is structured.

In [1]:
import pandas as pd

from functools import reduce

## Defining NHANES Cycles and Data Files

This step defines the NHANES survey cycles and component files used to build the analysis-ready dataset. Each cycle contains a base URL pointing to the CDC NHANES data directory and a list of selected `.xpt` files that contain the variables needed for this study.

The selected files cover the main data components required for diabetes status prediction: demographics, body measurements, blood pressure, diabetes questionnaire data, smoking, alcohol use, physical activity, sleep, laboratory measurements, and dietary nutrient intake.

Each NHANES cycle uses a different file suffix. For example, the 2011--2012 cycle uses `_G`, 2013--2014 uses `_H`, and 2015--2016 uses `_I`. The 2017--March 2020 pre-pandemic release uses the `P_` prefix instead of the regular cycle suffix. These file mappings are defined explicitly so the data can be downloaded and merged programmatically for each cycle.



| Cycle | File suffix / prefix | Notes |
|---|---|---|
| 2011--2012 | `_G` | Standard NHANES cycle |
| 2013--2014 | `_H` | Standard NHANES cycle |
| 2015--2016 | `_I` | Standard NHANES cycle |
| 2017--March 2020 | `P_` | Special pre-pandemic release |


| Module | Purpose |
|---|---|
| `DEMO` | Demographic variables such as age, sex, race/ethnicity, education, and income |
| `BMX` | Body measurements such as BMI and waist circumference |
| `BPX` / `BPXO` | Blood pressure measurements |
| `DIQ` | Diabetes questionnaire variables used for outcome construction |
| `SMQ` | Smoking-related variables |
| `ALQ` | Alcohol-use variables |
| `PAQ` | Physical activity variables |
| `SLQ` | Sleep-related variables |
| `GHB` | HbA1c / glycohemoglobin laboratory values |
| `GLU` | Fasting glucose laboratory values |
| `DR1TOT` | First-day dietary nutrient intake totals |

In [58]:
CYCLES = {

    "2011-2012": {

        "base": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2011/DataFiles",

        "files": {

            "DEMO": "DEMO_G.xpt",

            "BMX": "BMX_G.xpt",

            "BPX": "BPX_G.xpt",

            "DIQ": "DIQ_G.xpt",

            "SMQ": "SMQ_G.xpt",

            "ALQ": "ALQ_G.xpt",

            "PAQ": "PAQ_G.xpt",

            "SLQ": "SLQ_G.xpt",

            "GHB": "GHB_G.xpt",

            "GLU": "GLU_G.xpt",

            "DR1TOT": "DR1TOT_G.xpt",

        },

    },

    "2013-2014": {

        "base": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles",

        "files": {

            "DEMO": "DEMO_H.xpt",

            "BMX": "BMX_H.xpt",

            "BPX": "BPX_H.xpt",

            "DIQ": "DIQ_H.xpt",

            "SMQ": "SMQ_H.xpt",

            "ALQ": "ALQ_H.xpt",

            "PAQ": "PAQ_H.xpt",

            "SLQ": "SLQ_H.xpt",

            "GHB": "GHB_H.xpt",

            "GLU": "GLU_H.xpt",

            "DR1TOT": "DR1TOT_H.xpt",

        },

    },

    "2015-2016": {

        "base": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles",

        "files": {

            "DEMO": "DEMO_I.xpt",

            "BMX": "BMX_I.xpt",

            "BPX": "BPX_I.xpt",

            "DIQ": "DIQ_I.xpt",

            "SMQ": "SMQ_I.xpt",

            "ALQ": "ALQ_I.xpt",

            "PAQ": "PAQ_I.xpt",

            "SLQ": "SLQ_I.xpt",

            "GHB": "GHB_I.xpt",

            "GLU": "GLU_I.xpt",

            "DR1TOT": "DR1TOT_I.xpt",

        },

    },

    "2017-March 2020": {

        "base": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles",

        "files": {

            "DEMO": "P_DEMO.xpt",

            "BMX": "P_BMX.xpt",

            "BPXO": "P_BPXO.xpt",

            "DIQ": "P_DIQ.xpt",

            "SMQ": "P_SMQ.xpt",

            "ALQ": "P_ALQ.xpt",

            "PAQ": "P_PAQ.xpt",

            "SLQ": "P_SLQ.xpt",

            "GHB": "P_GHB.xpt",

            "GLU": "P_GLU.xpt",

            "DR1TOT": "P_DR1TOT.xpt",

        },

    },

}

## Defining Required Variables and Harmonization Groups

This step defines the NHANES variables that are required to build the analysis-ready dataset. The `REQUIRED_COLUMNS` list contains variables that should be available across the selected NHANES cycles, including participant identifiers, demographic variables, anthropometric measures, diabetes outcome variables, lifestyle variables, laboratory measurements, and dietary intake variables.

Some variables differ between NHANES cycles and are therefore handled separately. Sleep duration may be stored under different names, so `SLEEP_ALTERNATIVES` lists the possible sleep variables that can be used to create one harmonized `sleep_hours` feature.

Blood pressure variables also differ across cycles. Older NHANES cycles use the standard systolic and diastolic blood pressure variables, while the 2017--March 2020 pre-pandemic cycle may use oscillometric blood pressure variables. These groups are defined separately so that systolic and diastolic blood pressure can later be harmonized into consistent features: `mean_systolic_bp` and `mean_diastolic_bp`.

## Required NHANES Variables

The table below summarizes the NHANES variables used in this project and explains what each code represents.

| NHANES variable | Meaning | Role in this project |
|---|---|---|
| `SEQN` | Participant identifier | Used to merge files across NHANES components |
| `RIDAGEYR` | Age in years | Predictor |
| `RIAGENDR` | Sex | Predictor |
| `RIDRETH3` | Race/ethnicity | Predictor |
| `DMDEDUC2` | Education level | Predictor |
| `INDFMPIR` | Family income-to-poverty ratio | Predictor |
| `RIDEXPRG` | Pregnancy status | Used for exclusion criteria |
| `BMXBMI` | Body mass index | Predictor |
| `BMXWAIST` | Waist circumference | Predictor |
| `DIQ010` | Doctor told participant they have diabetes | Used to construct outcome |
| `DID040` | Age when first told they had diabetes | Possible type 1 diabetes exclusion / descriptive variable |
| `DIQ050` | Taking insulin | Used to construct outcome |
| `DIQ070` | Taking diabetic pills | Used to construct outcome |
| `SMQ020` | Smoked at least 100 cigarettes in life | Predictor |
| `ALQ130` | Average number of alcoholic drinks per drinking day | Predictor |
| `PAQ605` | Vigorous work activity | Predictor |
| `PAQ620` | Moderate work activity | Predictor |
| `PAQ635` | Walking or bicycling for transportation | Predictor |
| `PAQ650` | Vigorous recreational activity | Predictor |
| `PAQ665` | Moderate recreational activity | Predictor |
| `LBXGH` | HbA1c / glycohemoglobin | Used to construct outcome |
| `LBXGLU` | Fasting glucose in mg/dL | Used to construct outcome |
| `LBDGLUSI` | Fasting glucose in mmol/L | Used to construct outcome / unit equivalent |
| `DR1TKCAL` | Total energy intake, first dietary recall day | Predictor |
| `DR1TPROT` | Protein intake, first dietary recall day | Predictor |
| `DR1TCARB` | Carbohydrate intake, first dietary recall day | Predictor |
| `DR1TSUGR` | Total sugar intake, first dietary recall day | Predictor |
| `DR1TFIBE` | Fiber intake, first dietary recall day | Predictor |
| `DR1TTFAT` | Total fat intake, first dietary recall day | Predictor |
| `DR1TCHOL` | Cholesterol intake, first dietary recall day | Predictor |

## Variables Requiring Harmonization Across Cycles

Some variables differ across NHANES cycles. These variables are checked separately and later harmonized into consistent analysis variables.

| Group | Possible NHANES variables | Harmonized variable | Meaning |
|---|---|---|---|
| Sleep duration | `SLD010H`, `SLD012` | `sleep_hours` | Sleep duration in hours |
| Standard systolic blood pressure | `BPXSY1`, `BPXSY2`, `BPXSY3` | `mean_systolic_bp` | Mean systolic blood pressure from available readings |
| Standard diastolic blood pressure | `BPXDI1`, `BPXDI2`, `BPXDI3` | `mean_diastolic_bp` | Mean diastolic blood pressure from available readings |
| Oscillometric systolic blood pressure | `BPXOSY1`, `BPXOSY2`, `BPXOSY3` | `mean_systolic_bp` | Mean systolic blood pressure from available oscillometric readings |
| Oscillometric diastolic blood pressure | `BPXODI1`, `BPXODI2`, `BPXODI3` | `mean_diastolic_bp` | Mean diastolic blood pressure from available oscillometric readings |

In [59]:
REQUIRED_COLUMNS = [

    "SEQN",

    "RIDAGEYR", "RIAGENDR", "RIDRETH3", "DMDEDUC2", "INDFMPIR", "RIDEXPRG",

    "BMXBMI", "BMXWAIST",

    "DIQ010", "DID040", "DIQ050", "DIQ070",

    "SMQ020",

    "ALQ130",

    "PAQ605", "PAQ620", "PAQ635", "PAQ650", "PAQ665",

    "LBXGH",

    "LBXGLU", "LBDGLUSI",

    "DR1TKCAL", "DR1TPROT", "DR1TCARB", "DR1TSUGR",

    "DR1TFIBE", "DR1TTFAT", "DR1TCHOL"

]

SLEEP_ALTERNATIVES = ["SLD010H", "SLD012"]

BP_OLD_SYSTOLIC = ["BPXSY1", "BPXSY2", "BPXSY3"]

BP_OLD_DIASTOLIC = ["BPXDI1", "BPXDI2", "BPXDI3"]

BP_OSC_SYSTOLIC = ["BPXOSY1", "BPXOSY2", "BPXOSY3"]

BP_OSC_DIASTOLIC = ["BPXODI1", "BPXODI2", "BPXODI3"]

## Loading and Merging NHANES Data by Cycle

This step loads the selected NHANES data files for each survey cycle defined in the `CYCLES` dictionary. Each cycle contains a base URL and a set of NHANES component files, such as demographics, body measurements, blood pressure, diabetes questionnaire data, lifestyle variables, laboratory data, and dietary intake data.

For each cycle, the code iterates through the required files and reads the original NHANES `.XPT` files directly from the CDC website using `pandas.read_sas()`. Each file is checked for the presence of the respondent identifier `SEQN`, which is required to merge records belonging to the same participant. Files that cannot be loaded or do not contain `SEQN` are skipped with an error or warning message.

After all available files for a cycle are loaded, they are merged into a single dataframe using a left join on `SEQN`. This creates one combined dataset per cycle, where each row represents one NHANES participant and the columns contain variables from the selected NHANES components. A `CYCLE` column is then added to indicate the survey cycle from which each participant record originates.

The printed output shows the shape of each loaded file and the final merged dataset for each cycle. This helps verify that the files were loaded correctly and that the merge process produced the expected number of rows and columns.

In [60]:
merged_cycles = {}

for cycle in CYCLES:
    print(f"Loading data for cycle: {cycle}")
    
    base_url = CYCLES[cycle]["base"]
    files = CYCLES[cycle]["files"]
    
    dfs = []
    for key, filename in files.items():
        url = f"{base_url}/{filename}"
        
        try:
            temp = pd.read_sas(url, format="xport")

        except Exception as e:
            print(f"[ERROR] Could not load {key}: {filename}")
            print(f"URL: {url}")
            print(f"Error: {e}")
            continue
        
        if "SEQN" not in temp.columns:
            print(f"[WARNING] {key} does not contain SEQN, skipping.")
            continue
            
        print(f"{key} {filename} shape={temp.shape}")
        dfs.append(temp)
    
    if not dfs:
        raise ValueError(f"No files could be loaded for cycle {cycle}")
    
    # Merge all dataframes on SEQN
    merged_df = reduce(lambda left, right: pd.merge(left, right, on="SEQN", how="left"), dfs)
    merged_df["CYCLE"] = cycle
    
    print(f"\nMerged shape for {cycle}: {merged_df.shape}")
    merged_cycles[cycle] = merged_df

Loading data for cycle: 2011-2012
DEMO DEMO_G.xpt shape=(9756, 48)
BMX BMX_G.xpt shape=(9338, 26)
BPX BPX_G.xpt shape=(9338, 27)
DIQ DIQ_G.xpt shape=(9364, 53)
SMQ SMQ_G.xpt shape=(6790, 30)
ALQ ALQ_G.xpt shape=(5615, 10)
PAQ PAQ_G.xpt shape=(9107, 21)
SLQ SLQ_G.xpt shape=(6175, 4)
GHB GHB_G.xpt shape=(6549, 2)
GLU GLU_G.xpt shape=(3239, 8)
DR1TOT DR1TOT_G.xpt shape=(9338, 166)

Merged shape for 2011-2012: (9756, 386)
Loading data for cycle: 2013-2014
DEMO DEMO_H.xpt shape=(10175, 47)
BMX BMX_H.xpt shape=(9813, 26)
BPX BPX_H.xpt shape=(9813, 23)
DIQ DIQ_H.xpt shape=(9770, 54)
SMQ SMQ_H.xpt shape=(7168, 32)
ALQ ALQ_H.xpt shape=(5924, 10)
PAQ PAQ_H.xpt shape=(9484, 96)
SLQ SLQ_H.xpt shape=(6464, 4)
GHB GHB_H.xpt shape=(6979, 2)
GLU GLU_H.xpt shape=(3329, 6)
DR1TOT DR1TOT_H.xpt shape=(9813, 168)

Merged shape for 2013-2014: (10175, 459)
Loading data for cycle: 2015-2016
DEMO DEMO_I.xpt shape=(9971, 47)
BMX BMX_I.xpt shape=(9544, 26)
BPX BPX_I.xpt shape=(9544, 21)
DIQ DIQ_I.xpt shape=(9575

## Checking Required Columns Across NHANES Cycles

This step verifies whether the merged dataset for each NHANES cycle contains the variables required for the analysis. Most selected variables have consistent names across cycles and are checked using the `REQUIRED_COLUMNS` list.

Some variables differ between NHANES cycles and therefore require separate checks. Sleep duration may be stored under different variable names, such as `SLD010H` or `SLD012`, depending on the cycle. Blood pressure measurements also differ between older cycles and the 2017--March 2020 pre-pandemic cycle. Older cycles use the standard blood pressure variables, while the pre-pandemic cycle may use oscillometric blood pressure variables.

For each cycle, the code reports whether all standard required columns are present, which sleep variable is available, and whether a complete set of blood pressure variables can be found. This helps identify harmonization issues before constructing the final analysis dataset.

In [61]:
for cycle, df in merged_cycles.items():
    print(f"\nChecking cycle: {cycle}")
    print("-" * 80)

    missing_cols = [col for col in REQUIRED_COLUMNS if col not in df.columns]

    if missing_cols:
        print(f"[WARNING] Missing required columns: {missing_cols}")

    else:
        print("[OK] All standard required columns are present.")

    available_sleep = [col for col in SLEEP_ALTERNATIVES if col in df.columns]

    if available_sleep:
        print(f"[OK] Sleep column available: {available_sleep}")

    else:
        print(f"[WARNING] No sleep column found. Expected one of: {SLEEP_ALTERNATIVES}")

    has_old_bp = all(col in df.columns for col in BP_OLD_SYSTOLIC + BP_OLD_DIASTOLIC)
    has_osc_bp = all(col in df.columns for col in BP_OSC_SYSTOLIC + BP_OSC_DIASTOLIC)

    if has_old_bp:
        print("[OK] Blood pressure columns available: old BP format")
    elif has_osc_bp:
        print("[OK] Blood pressure columns available: oscillometric BP format")


    else:
        print("[WARNING] Complete blood pressure set not found.")


Checking cycle: 2011-2012
--------------------------------------------------------------------------------
[OK] All standard required columns are present.
[OK] Sleep column available: ['SLD010H']
[OK] Blood pressure columns available: old BP format

Checking cycle: 2013-2014
--------------------------------------------------------------------------------
[OK] All standard required columns are present.
[OK] Sleep column available: ['SLD010H']
[OK] Blood pressure columns available: old BP format

Checking cycle: 2015-2016
--------------------------------------------------------------------------------
[OK] All standard required columns are present.
[OK] Sleep column available: ['SLD012']
[OK] Blood pressure columns available: old BP format

Checking cycle: 2017-March 2020
--------------------------------------------------------------------------------
[OK] All standard required columns are present.
[OK] Sleep column available: ['SLD012']
[OK] Blood pressure columns available: oscillomet

In [62]:
nhanes_raw = pd.concat(merged_cycles.values(), ignore_index=True)

print(nhanes_raw.shape)
print(nhanes_raw["CYCLE"].value_counts())

(45462, 523)
CYCLE
2017-March 2020    15560
2013-2014          10175
2015-2016           9971
2011-2012           9756
Name: count, dtype: int64


## Harmonizing Sleep and Blood Pressure Variables

This step creates consistent sleep and blood pressure variables across the combined NHANES dataset. Because variable names and measurement formats differ between NHANES cycles, the raw variables first need to be harmonized before they can be used in the analysis dataset.

For sleep duration, the code checks the available sleep variables listed in `SLEEP_ALTERNATIVES` and selects the first non-missing value for each participant. The selected value is stored in a new standardized column called `sleep_hours`.

For blood pressure, NHANES uses different variable formats across cycles. Earlier cycles contain standard systolic and diastolic blood pressure readings, while the 2017--March 2020 pre-pandemic cycle may contain oscillometric blood pressure readings. The code checks which set of blood pressure variables is available for each participant and calculates the mean systolic and mean diastolic blood pressure using the non-missing readings. These values are stored as `mean_systolic_bp` and `mean_diastolic_bp`.

This harmonization step ensures that later modelling steps can use consistent feature names across all NHANES cycles, even when the original variable names differ between survey years.

In [63]:
def harmonize_sleep(row):
    for col in SLEEP_ALTERNATIVES:
        if col in row.index and pd.notna(row[col]):
            return row[col]
    return pd.NA

nhanes_raw["sleep_hours"] = nhanes_raw.apply(harmonize_sleep, axis=1)


def harmonize_blood_pressure(row):
    old_sys_available = [col for col in BP_OLD_SYSTOLIC if col in row.index and pd.notna(row[col])]
    old_dia_available = [col for col in BP_OLD_DIASTOLIC if col in row.index and pd.notna(row[col])]

    osc_sys_available = [col for col in BP_OSC_SYSTOLIC if col in row.index and pd.notna(row[col])]
    osc_dia_available = [col for col in BP_OSC_DIASTOLIC if col in row.index and pd.notna(row[col])]

    if old_sys_available or old_dia_available:
        systolic = row[old_sys_available].mean() if old_sys_available else pd.NA
        diastolic = row[old_dia_available].mean() if old_dia_available else pd.NA
        return pd.Series({"systolic": systolic, "diastolic": diastolic})

    if osc_sys_available or osc_dia_available:
        systolic = row[osc_sys_available].mean() if osc_sys_available else pd.NA
        diastolic = row[osc_dia_available].mean() if osc_dia_available else pd.NA
        return pd.Series({"systolic": systolic, "diastolic": diastolic})

    return pd.Series({"systolic": pd.NA, "diastolic": pd.NA})


bp_harmonized = nhanes_raw.apply(harmonize_blood_pressure, axis=1)

nhanes_raw["mean_systolic_bp"] = bp_harmonized["systolic"]
nhanes_raw["mean_diastolic_bp"] = bp_harmonized["diastolic"]

/var/folders/mp/s58yt3093cxbv6x658kqkjq40000gn/T/ipykernel_15560/2094595956.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  nhanes_raw["sleep_hours"] = nhanes_raw.apply(harmonize_sleep, axis=1)
/var/folders/mp/s58yt3093cxbv6x658kqkjq40000gn/T/ipykernel_15560/2094595956.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  nhanes_raw["mean_systolic_bp"] = bp_harmonized["systolic"]
/var/folders/mp/s58yt3093cxbv6x658kqkjq40000gn/T/ipykernel_15560/2094595956.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usua

## Constructing the Diabetes Outcome Variable

This step constructs the binary diabetes outcome variable used for the prediction task. Participants are classified as having diabetes if they meet at least one of several criteria: self-reported doctor diagnosis of diabetes, HbA1c of at least 6.5%, fasting plasma glucose of at least 126 mg/dL, insulin use, or diabetes medication use.

Each criterion is first converted into a Boolean condition. These conditions are then combined using a logical OR operation, meaning that a participant is labelled as diabetic if any of the criteria are true. The resulting outcome is stored in the `diabetes` column, where `1` indicates diabetes and `0` indicates non-diabetes.

Participants with no available information for any of the outcome-defining variables are assigned a missing value (`NA`) for the diabetes outcome. This prevents participants from being incorrectly labelled as non-diabetic when there is not enough information to determine their status.

The final output displays both the raw class counts and the normalized class distribution. This is used to inspect the balance between diabetic and non-diabetic participants before further preprocessing and modelling.

In [64]:
self_report = nhanes_raw["DIQ010"] == 1
hba1c = nhanes_raw["LBXGH"] >= 6.5
fasting_glucose = nhanes_raw["LBXGLU"] >= 126
insulin_use = nhanes_raw["DIQ050"] == 1
diabetes_medication = nhanes_raw["DIQ070"] == 1

has_outcome_info = (
    nhanes_raw["DIQ010"].notna()
    | nhanes_raw["LBXGH"].notna()
    | nhanes_raw["LBXGLU"].notna()
    | nhanes_raw["DIQ050"].notna()
    | nhanes_raw["DIQ070"].notna()
)

nhanes_raw["diabetes"] = (
    self_report
    | hba1c
    | fasting_glucose
    | insulin_use
    | diabetes_medication
).astype("Int64")

nhanes_raw.loc[~has_outcome_info, "diabetes"] = pd.NA

print(nhanes_raw["diabetes"].value_counts(dropna=False))
print(nhanes_raw["diabetes"].value_counts(normalize=True, dropna=True))

diabetes
0       38692
1        5001
<NA>     1769
Name: count, dtype: Int64
diabetes
0    0.885542
1    0.114458
Name: proportion, dtype: Float64


/var/folders/mp/s58yt3093cxbv6x658kqkjq40000gn/T/ipykernel_15560/3452679048.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  nhanes_raw["diabetes"] = (


## Applying Population Criteria

This step applies the inclusion and exclusion criteria for the analysis dataset. The study population is restricted to adult NHANES participants aged 20 years and older. Participants without sufficient information to define the diabetes outcome are excluded, because their diabetes status cannot be determined reliably.

Pregnant participants are also excluded where pregnancy status is available, since pregnancy can affect metabolic markers and may introduce a different clinical context. This step helps ensure that the final analysis dataset reflects the intended adult non-pregnant population for diabetes status prediction.

After applying these criteria, the class distribution of the diabetes outcome is checked again to verify the final number of diabetic and non-diabetic participants before feature selection and modelling.

In [65]:
analysis_df = nhanes_raw.copy()


analysis_df = analysis_df[analysis_df["diabetes"].notna()]
analysis_df = analysis_df[analysis_df["RIDAGEYR"] >= 20]

if "RIDEXPRG" in analysis_df.columns:
    analysis_df = analysis_df[analysis_df["RIDEXPRG"] != 1]

print(analysis_df.shape)

print(analysis_df["diabetes"].value_counts(dropna=False))

print(analysis_df["diabetes"].value_counts(normalize=True, dropna=True))

(26001, 527)
diabetes
0    21092
1     4909
Name: count, dtype: Int64
diabetes
0    0.8112
1    0.1888
Name: proportion, dtype: Float64


## Selecting and Renaming Analysis Variables

This step creates the final analysis dataset by selecting only the participant identifier, survey cycle, diabetes outcome, and predictor variables needed for modelling. The selected predictors include demographic, socioeconomic, anthropometric, blood pressure, lifestyle, sleep, and dietary variables.

The original NHANES variable names are then renamed to more interpretable column names. This makes the dataset easier to understand during preprocessing, model training, evaluation, and feature-importance analysis. The original NHANES codes are preserved in the `column_rename_map`, which documents how each raw NHANES variable corresponds to the renamed analysis variable.

Variables used to construct the diabetes outcome, such as HbA1c, fasting glucose, diabetes diagnosis, insulin use, and diabetes medication use, are not included as predictors. This prevents data leakage and ensures that the models use risk-related variables rather than direct diagnostic indicators.

In [66]:
predictor_columns = [
    "RIDAGEYR",
    "RIAGENDR",
    "RIDRETH3",
    "DMDEDUC2",
    "INDFMPIR",
    "BMXBMI",
    "BMXWAIST",
    "mean_systolic_bp",
    "mean_diastolic_bp",
    "SMQ020",
    "ALQ130",
    "PAQ605",
    "PAQ620",
    "PAQ635",
    "PAQ650",
    "PAQ665",
    "sleep_hours",
    "DR1TKCAL",
    "DR1TPROT",
    "DR1TCARB",
    "DR1TSUGR",
    "DR1TFIBE",
    "DR1TTFAT",
    "DR1TCHOL",
]

analysis_df = analysis_df[["SEQN", "CYCLE", "diabetes"] + predictor_columns].copy()

column_rename_map = {
    "SEQN": "participant_id",
    "CYCLE": "cycle",
    "RIDAGEYR": "age",
    "RIAGENDR": "sex",
    "RIDRETH3": "race_ethnicity",
    "DMDEDUC2": "education_level",
    "INDFMPIR": "income_poverty_ratio",
    "BMXBMI": "bmi",
    "BMXWAIST": "waist_circumference",
    "mean_systolic_bp": "mean_systolic_bp",
    "mean_diastolic_bp": "mean_diastolic_bp",
    "SMQ020": "ever_smoked_100_cigarettes",
    "ALQ130": "average_alcoholic_drinks_per_day",
    "PAQ605": "vigorous_work_activity",
    "PAQ620": "moderate_work_activity",
    "PAQ635": "walk_or_bicycle_transport",
    "PAQ650": "vigorous_recreational_activity",
    "PAQ665": "moderate_recreational_activity",
    "sleep_hours": "sleep_hours",
    "DR1TKCAL": "energy_kcal",
    "DR1TPROT": "protein_g",
    "DR1TCARB": "carbohydrate_g",
    "DR1TSUGR": "total_sugar_g",
    "DR1TFIBE": "fiber_g",
    "DR1TTFAT": "total_fat_g",
    "DR1TCHOL": "cholesterol_mg",
}

analysis_df = analysis_df.rename(columns=column_rename_map)

analysis_df.head()

,participant_id,cycle,diabetes,age,sex,race_ethnicity,education_level,income_poverty_ratio,bmi,waist_circumference,...,vigorous_recreational_activity,moderate_recreational_activity,sleep_hours,energy_kcal,protein_g,carbohydrate_g,total_sugar_g,fiber_g,total_fat_g,cholesterol_mg
0,62161.0,2011-2012,0,22.0,1.0,3.0,3.0,3.15,23.3,81.0,...,2.0,2.0,8.0,2969.0,104.68,359.59,109.09,18.6,123.81,328.0
3,62164.0,2011-2012,0,44.0,2.0,3.0,4.0,1.67,23.2,80.1,...,1.0,1.0,8.0,1115.0,73.13,91.67,32.29,9.5,51.54,207.0
8,62169.0,2011-2012,0,21.0,1.0,6.0,3.0,0.33,20.1,69.6,...,2.0,2.0,8.0,1831.0,77.46,297.51,78.19,4.3,34.61,205.0
11,62172.0,2011-2012,0,43.0,2.0,4.0,3.0,2.02,33.3,120.4,...,2.0,2.0,8.0,1845.0,57.43,192.82,58.56,2.8,42.02,160.0
13,62174.0,2011-2012,0,80.0,1.0,3.0,5.0,4.30,33.9,116.5,...,2.0,2.0,9.0,1178.0,39.35,146.72,84.63,12.0,49.75,468.0


In [67]:
nhanes_raw.to_parquet(
    "DATASETS/nhanes_2011_2020_raw.parquet",
    index=False
)

nhanes_raw.to_csv(
    "DATASETS/nhanes_2011_2020_raw.csv",
    index=False
)

analysis_df.to_parquet(
    "DATASETS/nhanes_diabetes_analysis_ready.parquet",
    index=False
)

analysis_df.to_csv(
    "DATASETS/nhanes_diabetes_analysis_ready.csv",
    index=False
)

